In [ ]:
# from google.colab import drive
# drive.mount('/content/drive')

In [ ]:
!pip install timm

In [1]:
print("Hello World")

Hello World


In [7]:
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader

from torchvision import transforms
from PIL import Image

from sklearn.metrics import accuracy_score, roc_auc_score
from sklearn.model_selection import train_test_split
from tqdm import tqdm

import timm

In [ ]:
PROJECT_ROOT = os.path.abspath(".")
BASE_PATH = os.path.join(PROJECT_ROOT, "data-20260408T055836Z-3-001", "data", "nih")
FILTERED_PATH = os.path.join(BASE_PATH, "filtered")
IMAGE_PATH = os.path.join(FILTERED_PATH, "images")
SPLIT_PATH = os.path.join(BASE_PATH, "splits")
MODEL_DIR = os.path.join(PROJECT_ROOT, "models")
BEST_MODEL_PATH = os.path.join(MODEL_DIR, "best_vit.pth")

train_csv = os.path.join(SPLIT_PATH, "train.csv")
val_csv = os.path.join(SPLIT_PATH, "val.csv")
test_csv = os.path.join(SPLIT_PATH, "test.csv")
metadata_csv = os.path.join(FILTERED_PATH, "metadata.csv")

print("Using data root:", BASE_PATH)
print("Images folder exists:", os.path.exists(IMAGE_PATH))
print("Splits folder exists:", os.path.exists(SPLIT_PATH))

Using data root: c:\Users\vijaykuv\Downloads\Practice Project\data-20260408T055836Z-3-001\data\nih
Images folder exists: True
Splits folder exists: False


In [18]:
if os.path.exists(train_csv) and os.path.exists(val_csv) and os.path.exists(test_csv):
    train_df = pd.read_csv(train_csv)
    val_df = pd.read_csv(val_csv)
    test_df = pd.read_csv(test_csv)
    print("Loaded existing split CSV files.")
else:
    full_df = pd.read_csv(metadata_csv)
    train_df, temp_df = train_test_split(full_df, test_size=0.3, random_state=42, shuffle=True)
    val_df, test_df = train_test_split(temp_df, test_size=0.5, random_state=42, shuffle=True)
    print("Split files not found. Created train/val/test splits from metadata.csv.")

print("Train:", len(train_df))
print("Val:", len(val_df))
print("Test:", len(test_df))

Split files not found. Created train/val/test splits from metadata.csv.
Train: 2003
Val: 429
Test: 430


In [19]:
if not (os.path.exists(train_csv) and os.path.exists(val_csv) and os.path.exists(test_csv)):
    os.makedirs(SPLIT_PATH, exist_ok=True)
    train_df.to_csv(train_csv, index=False)
    val_df.to_csv(val_csv, index=False)
    test_df.to_csv(test_csv, index=False)
    print("Saved split CSVs:")
    print("-", train_csv)
    print("-", val_csv)
    print("-", test_csv)
else:
    print("Split CSVs already exist. No new files were written.")

Saved split CSVs:
- c:\Users\vijaykuv\Downloads\Practice Project\data-20260408T055836Z-3-001\data\nih\splits\train.csv
- c:\Users\vijaykuv\Downloads\Practice Project\data-20260408T055836Z-3-001\data\nih\splits\val.csv
- c:\Users\vijaykuv\Downloads\Practice Project\data-20260408T055836Z-3-001\data\nih\splits\test.csv


In [20]:
class NIHDataset(Dataset):
    def __init__(self, dataframe, image_dir, transform=None):
        self.df = dataframe
        self.image_dir = image_dir
        self.transform = transform

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        img_path = os.path.join(self.image_dir, row["Image Index"])
        image = Image.open(img_path).convert("RGB")

        label = torch.tensor(row["label"], dtype=torch.float32)

        if self.transform:
            image = self.transform(image)

        return image, label

In [11]:
train_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(10),
    transforms.ToTensor(),
    transforms.Normalize([0.5, 0.5, 0.5],
                         [0.5, 0.5, 0.5])
])

val_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize([0.5, 0.5, 0.5],
                         [0.5, 0.5, 0.5])
])

In [21]:
train_dataset = NIHDataset(train_df, IMAGE_PATH, train_transform)
val_dataset = NIHDataset(val_df, IMAGE_PATH, val_transform)
test_dataset = NIHDataset(test_df, IMAGE_PATH, val_transform)

train_loader = DataLoader(train_dataset, batch_size=16, shuffle=True, num_workers=2)
val_loader = DataLoader(val_dataset, batch_size=16, shuffle=False, num_workers=2)
test_loader = DataLoader(test_dataset, batch_size=16, shuffle=False, num_workers=2)

In [13]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
device

device(type='cpu')

In [22]:
model = timm.create_model('vit_base_patch16_224', pretrained=True)

# Replace head
model.head = nn.Linear(model.head.in_features, 1)

model = model.to(device)

In [23]:
criterion = nn.BCEWithLogitsLoss()
optimizer = optim.Adam(model.parameters(), lr=3e-5)

In [24]:
def train_one_epoch(model, loader):
    model.train()
    running_loss = 0

    for images, labels in tqdm(loader):
        images = images.to(device)
        labels = labels.to(device).unsqueeze(1)

        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        running_loss += loss.item()

    return running_loss / len(loader)

In [25]:
def evaluate(model, loader):
    model.eval()
    all_labels = []
    all_probs = []

    with torch.no_grad():
        for images, labels in loader:
            images = images.to(device)
            labels = labels.to(device).unsqueeze(1)

            outputs = model(images)
            probs = torch.sigmoid(outputs)

            all_labels.extend(labels.cpu().numpy())
            all_probs.extend(probs.cpu().numpy())

    all_labels = np.array(all_labels)
    all_probs = np.array(all_probs)

    preds = (all_probs > 0.5).astype(int)

    acc = accuracy_score(all_labels, preds)
    auc = roc_auc_score(all_labels, all_probs)

    return acc, auc

In [ ]:
EPOCHS = 5
best_auc = 0
os.makedirs(MODEL_DIR, exist_ok=True)

for epoch in range(EPOCHS):
    train_loss = train_one_epoch(model, train_loader)
    val_acc, val_auc = evaluate(model, val_loader)

    print(f"\nEpoch {epoch+1}")
    print(f"Train Loss: {train_loss:.4f}")
    print(f"Val Acc: {val_acc:.4f}, Val AUC: {val_auc:.4f}")

    if val_auc > best_auc:
        best_auc = val_auc
        torch.save(model.state_dict(), BEST_MODEL_PATH)
        print(f"Saved best model to: {BEST_MODEL_PATH}")

  0%|          | 0/126 [00:00<?, ?it/s]

In [ ]:
model.load_state_dict(torch.load(BEST_MODEL_PATH, map_location=device))

test_acc, test_auc = evaluate(model, test_loader)

print("Loaded model from:", BEST_MODEL_PATH)
print("\nTest Accuracy:", test_acc)
print("Test AUC:", test_auc)